In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/Amazon Sale Report.csv
/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/Cloud Warehouse Compersion Chart.csv
/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/May-2022.csv
/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/P  L March 2021.csv
/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/International sale Report.csv
/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/Expense IIGF.csv
/kaggle/input/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data/Sale Report.csv
/kaggle/input/datasets/vinaykamble289/amazon-data1/ml_dataset.csv


In [2]:
import pandas as pd
import numpy as np

# Load
df = pd.read_csv("/kaggle/input/datasets/vinaykamble289/amazon-data1/ml_dataset.csv")

print("Initial shape:", df.shape)

# ─────────────────────────────────────────────
# 1. BASIC CLEANING
# ─────────────────────────────────────────────
df = df.drop_duplicates()

# Numeric conversions
num_cols = [
    "product_rating", "review_count",
    "buybox_price", "avg_offer_price",
    "min_offer_price", "max_offer_price",
    "seller_count", "fba_offer_count",
    "price_vs_avg_region",
    "seller_win_rate", "seller_total_wins"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing prices smartly
df["avg_offer_price"].fillna(df["buybox_price"], inplace=True)
df["min_offer_price"].fillna(df["buybox_price"], inplace=True)
df["max_offer_price"].fillna(df["buybox_price"], inplace=True)

# Fill ratings
df["product_rating"].fillna(df["product_rating"].median(), inplace=True)
df["review_count"].fillna(0, inplace=True)

# ─────────────────────────────────────────────
# 2. TIME FEATURES
# ─────────────────────────────────────────────
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek

# ─────────────────────────────────────────────
# 3. DELIVERY FEATURE EXTRACTION
# ─────────────────────────────────────────────
df["delivery_msg"] = df["delivery_msg"].astype(str)

df["is_free_delivery"] = df["delivery_msg"].str.contains("FREE", case=False).astype(int)
df["is_fast_delivery"] = df["delivery_msg"].str.contains("1|2|tomorrow|today", case=False).astype(int)

# ─────────────────────────────────────────────
# 4. COMPETITION GROUP (CRITICAL)
# ─────────────────────────────────────────────
group_cols = ["asin", "location", "timestamp"]

df["group_id"] = df[group_cols].astype(str).agg("_".join, axis=1)

# Price ranking inside group
df["price_rank"] = df.groupby("group_id")["buybox_price"].rank(method="dense")

# Min price per group
df["min_price_group"] = df.groupby("group_id")["buybox_price"].transform("min")

# Avg price per group
df["avg_price_group"] = df.groupby("group_id")["buybox_price"].transform("mean")

# Derived features
df["price_diff_from_min"] = df["buybox_price"] - df["min_price_group"]
df["price_vs_group_avg"] = df["buybox_price"] / df["avg_price_group"]

# ─────────────────────────────────────────────
# 5. FBA COMPETITION FEATURES
# ─────────────────────────────────────────────
df["fba_ratio"] = df.groupby("group_id")["buybox_is_fba"].transform("mean")

# ─────────────────────────────────────────────
# 6. SELLER STRENGTH FEATURES
# ─────────────────────────────────────────────
df["seller_win_rate"] = df["seller_win_rate"].fillna(0)

# Log scale (important)
df["seller_total_wins_log"] = np.log1p(df["seller_total_wins"])

# ─────────────────────────────────────────────
# 7. TARGET VARIABLE
# ─────────────────────────────────────────────
df["target"] = (df["buybox_winner"] == df["buybox_seller"]).astype(int)

# ─────────────────────────────────────────────
# 8. FINAL FEATURE SELECTION
# ─────────────────────────────────────────────
features = [
    "buybox_price",
    "price_rank",
    "price_diff_from_min",
    "price_vs_group_avg",

    "product_rating",
    "review_count",

    "seller_win_rate",
    "seller_total_wins_log",

    "buybox_is_fba",
    "fba_ratio",

    "seller_count",

    "is_free_delivery",
    "is_fast_delivery",

    "hour",
    "day_of_week"
]

target = "target"

df_model = df[features + [target]].dropna()

print("Final ML dataset:", df_model.shape)

df_model.to_csv("ml_ready_geo_buybox.csv", index=False)
print("✅ ML-ready dataset saved!")

Initial shape: (730, 29)
Final ML dataset: (730, 16)
✅ ML-ready dataset saved!


/tmp/ipykernel_55/3294096775.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["avg_offer_price"].fillna(df["buybox_price"], inplace=True)
/tmp/ipykernel_55/3294096775.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=

In [3]:
df = pd.read_csv('/kaggle/working/ml_ready_geo_buybox.csv')

In [4]:
df

,buybox_price,price_rank,price_diff_from_min,price_vs_group_avg,product_rating,review_count,seller_win_rate,seller_total_wins_log,buybox_is_fba,fba_ratio,seller_count,is_free_delivery,is_fast_delivery,hour,day_of_week,target
0,499.00,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1,1.0,0,1,0,14,5,1
1,5911.00,1.0,0.0,1.0,4.7,128.0,0.0041,1.386294,0,0.0,0,1,1,14,5,1
2,560.00,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1,1.0,0,1,0,14,5,1
3,429.00,1.0,0.0,1.0,4.7,10.0,0.0055,1.609438,1,1.0,0,1,1,14,5,1
4,1200.00,1.0,0.0,1.0,4.2,0.0,0.1658,4.804021,1,1.0,0,1,0,14,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,389.00,1.0,0.0,1.0,5.0,1.0,0.0712,3.970292,0,0.0,0,0,0,11,5,0
726,389.00,1.0,0.0,1.0,4.3,7.0,0.0562,3.737670,0,0.0,0,1,0,11,5,1
727,310.67,1.0,0.0,1.0,4.2,0.0,0.0260,2.995732,0,0.0,0,1,1,11,5,1
728,639.00,1.0,0.0,1.0,4.6,6.0,0.0110,2.197225,0,0.0,12,0,1,11,5,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730 entries, 0 to 729
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   buybox_price           730 non-null    float64
 1   price_rank             730 non-null    float64
 2   price_diff_from_min    730 non-null    float64
 3   price_vs_group_avg     730 non-null    float64
 4   product_rating         730 non-null    float64
 5   review_count           730 non-null    float64
 6   seller_win_rate        730 non-null    float64
 7   seller_total_wins_log  730 non-null    float64
 8   buybox_is_fba          730 non-null    int64  
 9   fba_ratio              730 non-null    float64
 10  seller_count           730 non-null    int64  
 11  is_free_delivery       730 non-null    int64  
 12  is_fast_delivery       730 non-null    int64  
 13  hour                   730 non-null    int64  
 14  day_of_week            730 non-null    int64  
 15  target

In [6]:
df.value_counts()

buybox_price  price_rank  price_diff_from_min  price_vs_group_avg  product_rating  review_count  seller_win_rate  seller_total_wins_log  buybox_is_fba  fba_ratio  seller_count  is_free_delivery  is_fast_delivery  hour  day_of_week  target
699.00        1.0         0.0                  1.0                 3.6             7.0           0.0260           2.995732               0              0.0        0             1                 0                 12    5            1         6
185.00        1.0         0.0                  1.0                 4.2             0.0           0.0274           3.044522               1              1.0        0             1                 0                 13    5            1         5
699.00        1.0         0.0                  1.0                 3.6             7.0           0.0260           2.995732               1              1.0        0             1                 0                 13    5            1         4
750.00        1.0         0.0

In [7]:
def generate_price_scenarios(df):
    rows = []

    for _, row in df.iterrows():
        base_price = row["buybox_price"]

        for factor in [0.9, 0.95, 1.0, 1.05, 1.1]:
            new_row = row.copy()
            new_row["simulated_price"] = base_price * factor
            rows.append(new_row)

    return pd.DataFrame(rows)

In [8]:
df_new = generate_price_scenarios(df)

In [9]:
df_new

,buybox_price,price_rank,price_diff_from_min,price_vs_group_avg,product_rating,review_count,seller_win_rate,seller_total_wins_log,buybox_is_fba,fba_ratio,seller_count,is_free_delivery,is_fast_delivery,hour,day_of_week,target,simulated_price
0,499.0,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1.0,1.0,0.0,1.0,0.0,14.0,5.0,1.0,449.10
0,499.0,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1.0,1.0,0.0,1.0,0.0,14.0,5.0,1.0,474.05
0,499.0,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1.0,1.0,0.0,1.0,0.0,14.0,5.0,1.0,499.00
0,499.0,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1.0,1.0,0.0,1.0,0.0,14.0,5.0,1.0,523.95
0,499.0,1.0,0.0,1.0,4.2,0.0,0.0274,3.044522,1.0,1.0,0.0,1.0,0.0,14.0,5.0,1.0,548.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,1799.0,1.0,0.0,1.0,4.2,0.0,0.0110,2.197225,0.0,0.0,0.0,0.0,1.0,11.0,5.0,1.0,1619.10
729,1799.0,1.0,0.0,1.0,4.2,0.0,0.0110,2.197225,0.0,0.0,0.0,0.0,1.0,11.0,5.0,1.0,1709.05
729,1799.0,1.0,0.0,1.0,4.2,0.0,0.0110,2.197225,0.0,0.0,0.0,0.0,1.0,11.0,5.0,1.0,1799.00
729,1799.0,1.0,0.0,1.0,4.2,0.0,0.0110,2.197225,0.0,0.0,0.0,0.0,1.0,11.0,5.0,1.0,1888.95


In [10]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3650 entries, 0 to 729
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   buybox_price           3650 non-null   float64
 1   price_rank             3650 non-null   float64
 2   price_diff_from_min    3650 non-null   float64
 3   price_vs_group_avg     3650 non-null   float64
 4   product_rating         3650 non-null   float64
 5   review_count           3650 non-null   float64
 6   seller_win_rate        3650 non-null   float64
 7   seller_total_wins_log  3650 non-null   float64
 8   buybox_is_fba          3650 non-null   float64
 9   fba_ratio              3650 non-null   float64
 10  seller_count           3650 non-null   float64
 11  is_free_delivery       3650 non-null   float64
 12  is_fast_delivery       3650 non-null   float64
 13  hour                   3650 non-null   float64
 14  day_of_week            3650 non-null   float64
 15  target    

In [11]:
df_new.describe()

,buybox_price,price_rank,price_diff_from_min,price_vs_group_avg,product_rating,review_count,seller_win_rate,seller_total_wins_log,buybox_is_fba,fba_ratio,seller_count,is_free_delivery,is_fast_delivery,hour,day_of_week,target,simulated_price
count,3650.000000,3650.0,3650.0,3650.0,3650.000000,3650.000000,3650.000000,3650.000000,3650.000000,3650.000000,3650.000000,3650.000000,3650.000000,3650.000000,3650.0,3650.000000,3650.000000
mean,870.482932,1.0,0.0,1.0,4.072329,7.036986,0.065304,3.346678,0.584932,0.584932,0.821918,0.723288,0.635616,12.871233,5.0,0.928767,870.482932
std,1549.759799,0.0,0.0,0.0,0.712994,21.501401,0.062019,1.099584,0.492801,0.492801,2.109255,0.447434,0.481323,0.834229,0.0,0.257249,1554.848530
min,81.000000,1.0,0.0,1.0,1.000000,0.000000,0.001400,0.693147,0.000000,0.000000,0.000000,0.000000,0.000000,11.000000,5.0,0.000000,72.900000
25%,200.000000,1.0,0.0,1.0,4.100000,0.000000,0.013700,2.397895,0.000000,0.000000,0.000000,0.000000,0.000000,12.000000,5.0,1.000000,200.000000
50%,368.000000,1.0,0.0,1.0,4.200000,1.000000,0.027400,3.044522,1.000000,1.000000,0.000000,1.000000,1.000000,13.000000,5.0,1.000000,351.500000
75%,699.000000,1.0,0.0,1.0,4.300000,5.000000,0.141100,4.644391,1.000000,1.000000,0.000000,1.000000,1.000000,14.000000,5.0,1.000000,697.500000
max,11233.810000,1.0,0.0,1.0,5.000000,200.000000,0.165800,4.804021,1.000000,1.000000,14.000000,1.000000,1.000000,14.000000,5.0,1.000000,12357.191000


In [48]:
import torch
import torch.nn as nn

class PriceOptimizerTS(nn.Module):
    def __init__(self, tabular_dim, seq_len):

        super().__init__()

        # 🔥 LSTM for time-series (price history)
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True
        )

        # 🔥 Tabular branch
        self.tabular_net = nn.Sequential(
            nn.Linear(tabular_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # 🔥 Combined network
        self.final_net = nn.Sequential(
            nn.Linear(32 + 32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, tabular_x, seq_x):

        # LSTM branch
        lstm_out, _ = self.lstm(seq_x)
        lstm_out = lstm_out[:, -1, :]   # last timestep

        # Tabular branch
        tab_out = self.tabular_net(tabular_x)

        # Combine
        combined = torch.cat([tab_out, lstm_out], dim=1)

        return self.final_net(combined)

In [49]:
def find_optimal_price(model, row, cost):
    prices = np.linspace(row["buybox_price"] * 0.8,
                         row["buybox_price"] * 1.2,
                         20)

    best_price = None
    best_profit = -1

    for p in prices:
        row["simulated_price"] = p

        prob = model.predict(row)

        expected_profit = prob * (p - cost)

        if expected_profit > best_profit:
            best_profit = expected_profit
            best_price = p

    return best_price, best_profit

In [50]:
MIN_PROFIT_MARGIN = 0.15
MIN_WIN_PROB = 0.6

In [51]:
# Add simulated price as feature
features = [
    "simulated_price",   # 🔥 NEW (most important)

    "price_rank",
    "price_diff_from_min",
    "price_vs_group_avg",

    "product_rating",
    "review_count",

    "seller_win_rate",
    "seller_total_wins_log",

    "buybox_is_fba",
    "fba_ratio",

    "seller_count",

    "is_free_delivery",
    "is_fast_delivery",

    "hour",
    "day_of_week"
]

target = "target"

In [52]:
from sklearn.model_selection import train_test_split

X = df_new[features]
y = df_new[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [53]:
X_test.loc[0]

simulated_price          449.100000
price_rank                 1.000000
price_diff_from_min        0.000000
price_vs_group_avg         1.000000
product_rating             4.200000
review_count               0.000000
seller_win_rate            0.027400
seller_total_wins_log      3.044522
buybox_is_fba              1.000000
fba_ratio                  1.000000
seller_count               0.000000
is_free_delivery           1.000000
is_fast_delivery           0.000000
hour                      14.000000
day_of_week                5.000000
Name: 0, dtype: float64

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [55]:
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [56]:
model = PriceOptimizerNN(input_dim=X_train_tensor.shape[1])

criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.6813
Epoch 1, Loss: 0.6660
Epoch 2, Loss: 0.6519
Epoch 3, Loss: 0.6391
Epoch 4, Loss: 0.6267
Epoch 5, Loss: 0.6143
Epoch 6, Loss: 0.6017
Epoch 7, Loss: 0.5886
Epoch 8, Loss: 0.5750
Epoch 9, Loss: 0.5609
Epoch 10, Loss: 0.5462
Epoch 11, Loss: 0.5309
Epoch 12, Loss: 0.5150
Epoch 13, Loss: 0.4984
Epoch 14, Loss: 0.4811
Epoch 15, Loss: 0.4632
Epoch 16, Loss: 0.4447
Epoch 17, Loss: 0.4257
Epoch 18, Loss: 0.4064
Epoch 19, Loss: 0.3869


In [57]:
model.eval()

with torch.no_grad():
    preds = model(X_test_tensor)
    preds = (preds > 0.5).float()

accuracy = (preds == y_test_tensor).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 0.9369863271713257


In [58]:
def predict_prob(model, scaler, row):

    # Convert to DataFrame
    row_df = pd.DataFrame([row])

    # 🔥 FORCE SAME FEATURES + ORDER
    row_df = row_df.reindex(columns=FEATURE_COLUMNS, fill_value=0)

    # Scale
    row_scaled = scaler.transform(row_df)

    row_tensor = torch.tensor(row_scaled, dtype=torch.float32)

    with torch.no_grad():
        prob = model(row_tensor).item()

    return prob

In [59]:
def find_optimal_price(model, scaler, row, cost):

    base_price = row["buybox_price"]

    prices = np.linspace(base_price * 0.8,
                         base_price * 1.2,
                         20)

    best_price = None
    best_profit = -1

    for p in prices:
        row_copy = row.copy()

        # 🔥 ADD simulated_price
        row_copy["simulated_price"] = p

        # 🔥 REMOVE unwanted columns
        for col in ["target"]:
            if col in row_copy:
                row_copy = row_copy.drop(col)

        prob = predict_prob(model, scaler, row_copy)

        expected_profit = prob * (p - cost)

        if expected_profit > best_profit:
            best_profit = expected_profit
            best_price = p

    return best_price, best_profit

In [60]:
FEATURE_COLUMNS = X_train.columns.tolist()

In [61]:
df_test = df_model.drop('target',axis=1)

In [62]:
row = df_model.iloc[0].copy()

best_price, profit = find_optimal_price(
    model,
    scaler,
    row,
    cost=400
)

print("Best Price:", best_price)
print("Profit:", profit)

Best Price: 598.8
Profit: 139.4822658538818


In [63]:
MIN_PROFIT_MARGIN = 0.15     # 15%
MIN_WIN_PROB = 0.6           # confidence threshold
MAX_PRICE_DROP = 0.2         # don’t reduce more than 20%

In [64]:
def pricing_decision(model, scaler, row, cost):

    best_price, expected_profit = find_optimal_price(
        model, scaler, row, cost
    )

    # Calculate metrics
    profit_margin = (best_price - cost) / cost

    # Get probability at best price
    row_copy = row.copy()
    row_copy["simulated_price"] = best_price

    if "target" in row_copy:
        row_copy = row_copy.drop("target")

    win_prob = predict_prob(model, scaler, row_copy)

    # Decision logic
    if win_prob < MIN_WIN_PROB:
        decision = "❌ Do NOT compete"

    elif profit_margin < MIN_PROFIT_MARGIN:
        decision = "⚠️ Low margin – risky"

    else:
        decision = "✅ Compete"

    return {
        "best_price": best_price,
        "expected_profit": expected_profit,
        "profit_margin": profit_margin,
        "win_probability": win_prob,
        "decision": decision
    }

In [65]:
row = df_model.iloc[0].copy()

result = pricing_decision(
    model,
    scaler,
    row,
    cost=400
)

print(result)

{'best_price': np.float64(598.8), 'expected_profit': np.float64(139.4822658538818), 'profit_margin': np.float64(0.4969999999999999), 'win_probability': 0.7016210556030273, 'decision': '✅ Compete'}


In [66]:
def strategy_adjustment(row, result):

    score = 0
    signals = []

    # 🔥 1. Competition signal
    if row["seller_count"] > 10:
        score -= 2
        signals.append("high competition")

    elif row["seller_count"] < 3:
        score += 2
        signals.append("low competition")

    # 🔥 2. FBA dominance
    if row["fba_ratio"] < 0.3:
        score += 2
        signals.append("low FBA dominance")

    elif row["fba_ratio"] > 0.7:
        score -= 1
        signals.append("FBA heavy market")

    # 🔥 3. Seller strength
    if row["seller_win_rate"] > 0.2:
        score += 2
        signals.append("strong seller")

    elif row["seller_win_rate"] < 0.05:
        score -= 1
        signals.append("weak seller")

    # 🔥 4. Model intelligence (MOST IMPORTANT)
    if result["win_probability"] > 0.7:
        score += 2
        signals.append("high win probability")

    elif result["win_probability"] < 0.4:
        score -= 2
        signals.append("low win probability")

    # 🔥 5. Profit signal
    if result["profit_margin"] > 0.25:
        score += 2
        signals.append("high margin")

    elif result["profit_margin"] < 0.1:
        score -= 2
        signals.append("low margin")

    # ─────────────────────────────
    # FINAL DECISION
    # ─────────────────────────────

    if score >= 4:
        strategy = "🚀 Aggressive – increase price"

    elif score >= 1:
        strategy = "✅ Balanced – compete normally"

    elif score >= -2:
        strategy = "⚠️ Defensive – reduce price carefully"

    else:
        strategy = "❌ Avoid – market unfavorable"

    return {
        "strategy": strategy,
        "score": score,
        "signals": signals
    }

In [67]:
result = pricing_decision(model, scaler, row, cost=550)

strategy = strategy_adjustment(row, result)

print("Decision:", result["decision"])
print("Strategy:", strategy)

Decision: ⚠️ Low margin – risky
Strategy: {'strategy': '⚠️ Defensive – reduce price carefully', 'score': 0, 'signals': ['low competition', 'FBA heavy market', 'weak seller', 'high win probability', 'low margin']}


In [68]:
import joblib
import torch

joblib.dump(scaler, "scaler.pkl")
torch.save(model.state_dict(), "model.pth")